# Day 39 – Q1: Hard NLP Patterns in Indian E-Commerce Reviews

Pipeline for 5 challenging patterns: Negation, Sarcasm, Code-mixing, Implicit, Comparative.
Each section shows baseline failure + feature engineering fix.

In [5]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

REVIEWS_PATH = "shopsense_reviews.csv"

NEGATION_WORDS = {"not", "never", "no", "neither", "nor", "n't", "nothing", "nowhere", "nobody"}
SARCASM_SIGNALS = ["wow", "great job", "amazing", "wonderful", "fantastic"]
HINDI_SENTIMENT_MAP = {
    "accha": "good", "bahut": "very", "kharab": "bad", "bekar": "useless",
    "zabardast": "excellent", "bakwas": "rubbish", "sahi": "right",
    "late": "late", "jaldi": "fast", "slow": "slow"
}
IMPLICIT_NEGATIVE = ["returned", "refund", "replacement", "waiting", "broken", "stopped working", "gave up"]
IMPLICIT_POSITIVE = ["repurchase", "gifted to", "ordered again", "shared with", "recommended"]
COMPARATIVE_POSITIVE = ["better than", "way better", "much better", "superior to", "prefer over"]
COMPARATIVE_NEGATIVE = ["worse than", "inferior to", "much worse", "not as good as"]


In [6]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        required = {"review_text", "sentiment_label", "language"}
        assert required.issubset(df.columns)
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")
    except AssertionError:
        raise ValueError(f"Missing required columns. Need: {required}")

df = load_reviews(REVIEWS_PATH)
vader = SentimentIntensityAnalyzer()
print("Dataset loaded:", df.shape)


Dataset loaded: (10199, 20)


## (a) Negation — 'not bad at all' is Positive

In [7]:
NEGATION_EXAMPLES = [
    "not bad at all",
    "not a good product",
    "not worth the money",
    "not disappointed with this purchase",
]

def baseline_vader(text):
    score = vader.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    return "Neutral"

def detect_negation_scope(text, negation_words=NEGATION_WORDS, window=3):
    tokens = text.lower().split()
    flipped = []
    negate_next = 0
    for tok in tokens:
        if tok in negation_words:
            negate_next = window
            flipped.append(tok)
        elif negate_next > 0:
            flipped.append("NOT_" + tok)
            negate_next -= 1
        else:
            flipped.append(tok)
    return " ".join(flipped)

print("=== Negation Analysis ===")
for text in NEGATION_EXAMPLES:
    processed = detect_negation_scope(text)
    baseline = baseline_vader(text)
    print(f"  Input     : {text}")
    print(f"  Processed : {processed}")
    print(f"  Baseline  : {baseline} (may be wrong)")
    print()

print("Baseline failure: VADER scores 'not bad' as Negative because 'bad' dominates.")
print("Fix: prepend NOT_ to tokens in negation scope before sentiment scoring.")


=== Negation Analysis ===
  Input     : not bad at all
  Processed : not NOT_bad NOT_at NOT_all
  Baseline  : Positive (may be wrong)

  Input     : not a good product
  Processed : not NOT_a NOT_good NOT_product
  Baseline  : Negative (may be wrong)

  Input     : not worth the money
  Processed : not NOT_worth NOT_the NOT_money
  Baseline  : Negative (may be wrong)

  Input     : not disappointed with this purchase
  Processed : not NOT_disappointed NOT_with NOT_this purchase
  Baseline  : Positive (may be wrong)

Baseline failure: VADER scores 'not bad' as Negative because 'bad' dominates.
Fix: prepend NOT_ to tokens in negation scope before sentiment scoring.


## (b) Sarcasm — 'Wow great! Broke on day 1'

In [8]:
SARCASM_EXAMPLES = [
    "Wow great! Broke on day 1.",
    "Amazing product. Stopped working after 2 days.",
    "Fantastic quality! Arrived broken.",
    "Really happy with this. Not.",
]

def extract_sarcasm_features(text, sarcasm_signals=SARCASM_SIGNALS):
    text_lower = text.lower()
    exclamation_count = text.count("!")
    caps_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
    has_signal_word = any(sig in text_lower for sig in sarcasm_signals)
    sentences = re.split(r"[.!?]", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    sentiment_scores = [vader.polarity_scores(s)["compound"] for s in sentences]
    has_sentiment_flip = (
        len(sentiment_scores) >= 2
        and sentiment_scores[0] > 0.2
        and sentiment_scores[-1] < -0.2
    )
    sarcasm_likely = has_signal_word and has_sentiment_flip
    return {
        "text": text,
        "exclamation_count": exclamation_count,
        "caps_ratio": round(caps_ratio, 3),
        "has_signal_word": has_signal_word,
        "has_sentiment_flip": has_sentiment_flip,
        "sarcasm_likely": sarcasm_likely,
        "baseline_vader": baseline_vader(text)
    }

print("=== Sarcasm Detection ===")
for ex in SARCASM_EXAMPLES:
    result = extract_sarcasm_features(ex)
    for k, v in result.items():
        print(f"  {k:22s}: {v}")
    print()

print("Baseline failure: VADER averages positive first sentence and negative second.")
print("Overall compound often comes out neutral, missing the sarcasm entirely.")
print("Fix: detect sentiment flip across sentences + signal-word presence as a sarcasm flag.")


=== Sarcasm Detection ===
  text                  : Wow great! Broke on day 1.
  exclamation_count     : 1
  caps_ratio            : 0.077
  has_signal_word       : True
  has_sentiment_flip    : True
  sarcasm_likely        : True
  baseline_vader        : Positive

  text                  : Amazing product. Stopped working after 2 days.
  exclamation_count     : 0
  caps_ratio            : 0.043
  has_signal_word       : True
  has_sentiment_flip    : True
  sarcasm_likely        : True
  baseline_vader        : Positive

  text                  : Fantastic quality! Arrived broken.
  exclamation_count     : 1
  caps_ratio            : 0.059
  has_signal_word       : True
  has_sentiment_flip    : True
  sarcasm_likely        : True
  baseline_vader        : Positive

  text                  : Really happy with this. Not.
  exclamation_count     : 0
  caps_ratio            : 0.071
  has_signal_word       : False
  has_sentiment_flip    : False
  sarcasm_likely        : False
  baselin

## (c) Code-mixing — Hindi-English reviews

In [9]:
CODE_MIXED_EXAMPLES = [
    "Product bahut accha hai lekin delivery late thi",
    "Zabardast quality! Bilkul bakwas packaging.",
    "Mujhe ye product bahut pasand aaya, kharab nahi hai.",
    "Ye product sahi hai, fast delivery.",
]

def normalize_code_mixed(text, hindi_map=HINDI_SENTIMENT_MAP):
    tokens = text.lower().split()
    translated = [hindi_map.get(tok, tok) for tok in tokens]
    return " ".join(translated)

def classify_code_mixed(text, hindi_map=HINDI_SENTIMENT_MAP):
    baseline = baseline_vader(text)
    normalized = normalize_code_mixed(text, hindi_map)
    fixed = baseline_vader(normalized)
    return {
        "original": text,
        "normalized": normalized,
        "baseline_pred": baseline,
        "fixed_pred": fixed
    }

print("=== Code-Mixed Review Analysis ===")
for ex in CODE_MIXED_EXAMPLES:
    result = classify_code_mixed(ex)
    for k, v in result.items():
        print(f"  {k:18s}: {v}")
    print()

print("Baseline failure: Hindi sentiment words are OOV for VADER — they get score=0.")
print("Fix: transliterate known Hindi sentiment words to English equivalents before scoring.")


=== Code-Mixed Review Analysis ===
  original          : Product bahut accha hai lekin delivery late thi
  normalized        : product very good hai lekin delivery late thi
  baseline_pred     : Neutral
  fixed_pred        : Positive

  original          : Zabardast quality! Bilkul bakwas packaging.
  normalized        : excellent quality! bilkul rubbish packaging.
  baseline_pred     : Neutral
  fixed_pred        : Positive

  original          : Mujhe ye product bahut pasand aaya, kharab nahi hai.
  normalized        : mujhe ye product very pasand aaya, bad nahi hai.
  baseline_pred     : Neutral
  fixed_pred        : Negative

  original          : Ye product sahi hai, fast delivery.
  normalized        : ye product right hai, fast delivery.
  baseline_pred     : Neutral
  fixed_pred        : Neutral

Baseline failure: Hindi sentiment words are OOV for VADER — they get score=0.
Fix: transliterate known Hindi sentiment words to English equivalents before scoring.


## (d) Implicit Sentiment — 'Returned it within 2 hours'

In [10]:
IMPLICIT_EXAMPLES = [
    "Returned it within 2 hours",
    "Had to ask for a replacement.",
    "Ordered again for my mother.",
    "Gave up after waiting 10 days.",
    "Gifted it to my colleague, she loved it.",
]

def detect_implicit_sentiment(
    text,
    neg_signals=IMPLICIT_NEGATIVE,
    pos_signals=IMPLICIT_POSITIVE
):
    text_lower = text.lower()
    neg_hit = next((s for s in neg_signals if s in text_lower), None)
    pos_hit = next((s for s in pos_signals if s in text_lower), None)
    if neg_hit and not pos_hit:
        implicit = "Negative"
        signal = neg_hit
    elif pos_hit and not neg_hit:
        implicit = "Positive"
        signal = pos_hit
    else:
        implicit = "Unknown"
        signal = None
    return {
        "text": text,
        "baseline_vader": baseline_vader(text),
        "implicit_pred": implicit,
        "trigger_signal": signal
    }

print("=== Implicit Sentiment Detection ===")
for ex in IMPLICIT_EXAMPLES:
    result = detect_implicit_sentiment(ex)
    for k, v in result.items():
        print(f"  {k:18s}: {v}")
    print()

print("Baseline failure: 'Returned it within 2 hours' has no explicit polarity word.")
print("VADER scores this as Neutral. Fix: behavioural keyword signals encode implicit polarity.")


=== Implicit Sentiment Detection ===
  text              : Returned it within 2 hours
  baseline_vader    : Neutral
  implicit_pred     : Negative
  trigger_signal    : returned

  text              : Had to ask for a replacement.
  baseline_vader    : Neutral
  implicit_pred     : Negative
  trigger_signal    : replacement

  text              : Ordered again for my mother.
  baseline_vader    : Neutral
  implicit_pred     : Positive
  trigger_signal    : ordered again

  text              : Gave up after waiting 10 days.
  baseline_vader    : Neutral
  implicit_pred     : Negative
  trigger_signal    : waiting

  text              : Gifted it to my colleague, she loved it.
  baseline_vader    : Positive
  implicit_pred     : Unknown
  trigger_signal    : None

Baseline failure: 'Returned it within 2 hours' has no explicit polarity word.
VADER scores this as Neutral. Fix: behavioural keyword signals encode implicit polarity.


## (e) Comparative — 'Way better than my previous Samsung'

In [11]:
COMPARATIVE_EXAMPLES = [
    "Way better than my previous Samsung",
    "Much worse than expected, prefer the old model",
    "This is better than everything I have used before",
    "Not as good as the previous version",
]

def extract_comparative_sentiment(
    text,
    pos_patterns=COMPARATIVE_POSITIVE,
    neg_patterns=COMPARATIVE_NEGATIVE
):
    text_lower = text.lower()
    pos_hit = next((p for p in pos_patterns if p in text_lower), None)
    neg_hit = next((p for p in neg_patterns if p in text_lower), None)
    if pos_hit:
        pred, trigger = "Positive", pos_hit
    elif neg_hit:
        pred, trigger = "Negative", neg_hit
    else:
        pred, trigger = baseline_vader(text), "no comparative pattern"
    return {
        "text": text,
        "baseline_vader": baseline_vader(text),
        "comparative_pred": pred,
        "trigger_pattern": trigger
    }

print("=== Comparative Sentiment Detection ===")
for ex in COMPARATIVE_EXAMPLES:
    result = extract_comparative_sentiment(ex)
    for k, v in result.items():
        print(f"  {k:20s}: {v}")
    print()

print("Baseline failure: VADER misses direction in 'Way better than X'.")
print("The reference brand name 'Samsung' carries no intrinsic sentiment.")
print("Fix: detect comparative phrase patterns and use direction to assign polarity.")


=== Comparative Sentiment Detection ===
  text                : Way better than my previous Samsung
  baseline_vader      : Positive
  comparative_pred    : Positive
  trigger_pattern     : better than

  text                : Much worse than expected, prefer the old model
  baseline_vader      : Negative
  comparative_pred    : Negative
  trigger_pattern     : worse than

  text                : This is better than everything I have used before
  baseline_vader      : Positive
  comparative_pred    : Positive
  trigger_pattern     : better than

  text                : Not as good as the previous version
  baseline_vader      : Negative
  comparative_pred    : Negative
  trigger_pattern     : not as good as

Baseline failure: VADER misses direction in 'Way better than X'.
The reference brand name 'Samsung' carries no intrinsic sentiment.
Fix: detect comparative phrase patterns and use direction to assign polarity.


## Pipeline: Apply All 5 Patterns on Real Reviews

In [12]:
def run_full_pipeline(text):
    result = {"text": text, "baseline": baseline_vader(text)}
    result["negation_processed"] = detect_negation_scope(text)
    result["sarcasm_features"] = extract_sarcasm_features(text)
    result["code_mixed_fixed"] = classify_code_mixed(text)["fixed_pred"]
    result["implicit_signals"] = detect_implicit_sentiment(text)
    result["comparative_signals"] = extract_comparative_sentiment(text)
    flags = []
    if result["sarcasm_features"]["sarcasm_likely"]:
        flags.append("SARCASM")
    if result["implicit_signals"]["implicit_pred"] != "Unknown":
        flags.append("IMPLICIT")
    if result["comparative_signals"]["trigger_pattern"] != "no comparative pattern":
        flags.append("COMPARATIVE")
    result["active_patterns"] = flags if flags else ["standard"]
    return result

sample_reviews = df[df["language"].isin(["English", "Code-mixed"])]["review_text"].dropna().head(5).tolist()
print("=== Full Pipeline on Sample Reviews ===")
for text in sample_reviews:
    r = run_full_pipeline(str(text))
    print(f"  Text    : {r['text'][:80]}")
    print(f"  Baseline: {r['baseline']} | Patterns: {r['active_patterns']}")
    print()


=== Full Pipeline on Sample Reviews ===
  Text    : <p>DO NOT BUY THIS</p>. Fake product. Nothing like the images shown on the websi
  Baseline: Positive | Patterns: ['standard']

  Text    : <p>DO NOT BUY THIS</p>. Fake product. Nothing like the images shown on the websi
  Baseline: Positive | Patterns: ['standard']

  Text    : Amazing value for money. The technical exceeded my expectations in every way pos
  Baseline: Positive | Patterns: ['standard']

  Text    : Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended
  Baseline: Positive | Patterns: ['IMPLICIT']

  Text    : Product bahut accha hai. Quality is top notch. Will buy again for sure.
  Baseline: Positive | Patterns: ['standard']

